In [1]:
import pandas as pd
import numpy as np
from causalml.inference.meta import BaseRRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder

In [55]:
order_info = pd.read_csv('order_info.csv')
order_info

,dt,stat_pay_user_id,stat_pay_main_order_id,category_name_cn,base_sku_name_en,brand_name_en_new,base_sku_id,sku_sale_amt,sku_sale_num,biz_total_discount_amt
0,20250920,User81,Order91,冷冻炸鸡肉,brand25 chicken nuggets 750g,Brand25,SKU100,34.905,1,7.650
1,20250903,User81,Order530,冷冻炸鸡肉,brand25 chicken nuggets 750g,Brand25,SKU100,30.930,1,6.750
2,20251101,User81,Order338,冷冻炸鸡肉,brand25 chicken nuggets 750g,Brand25,SKU100,38.925,1,6.000
3,20251007,User1615,Order12249,冷冻炸鸡肉,brand25 chicken nuggets 750g,Brand25,SKU100,36.675,1,8.250
4,20251130,User4406,Order23873,冷冻炸鸡肉,brand25 chicken nuggets 750g,Brand25,SKU100,30.150,1,14.775
...,...,...,...,...,...,...,...,...,...,...
206918,20251116,User14994,Order78134,袋装薯片/条,brand136 salt wavy potato chips 55g,Brand136,SKU999,12.750,2,0.000
206919,20251110,User14994,Order78916,袋装薯片/条,brand136 salt wavy potato chips 55g,Brand136,SKU999,19.125,3,0.000
206920,20251027,User15512,Order79607,袋装薯片/条,brand136 salt wavy potato chips 55g,Brand136,SKU999,12.750,2,0.000
206921,20250909,User15492,Order79822,袋装薯片/条,brand136 salt wavy potato chips 55g,Brand136,SKU999,4.470,1,0.000


In [56]:
activity_timeline = pd.read_csv('activity_timeline.csv')
activity_timeline

,日期,营销活动
0,2025/8/25,8月活动
1,2025/8/26,8月活动
2,2025/8/27,8月活动
3,2025/8/28,8月活动
4,2025/8/29,8月活动
5,2025/8/30,8月活动
6,2025/8/31,8月活动
7,2025/9/1,8月活动
8,2025/9/2,8月活动
9,2025/9/3,8月活动


In [57]:
# 日期格式变换
order_info['date'] = pd.to_datetime(order_info['dt'].astype(str), format='%Y%m%d')
activity_timeline['date'] = pd.to_datetime(activity_timeline['日期'].astype(str), format='%Y/%m/%d')

In [58]:
sku_daily_sales = order_info.groupby(['date', 'base_sku_id']).agg({
        'category_name_cn': 'first',      # 品类
        'brand_name_en_new': 'first',    # 品牌
        'sku_sale_amt': 'sum',           # 当日销售额
        'sku_sale_num': 'sum',           # 当日销量
        'biz_total_discount_amt': 'sum'  # 当日总优惠金额
    }).reset_index()

In [59]:
# 设定阈值
min_count = 30
df_cleaned = sku_daily_sales.copy()

print(f"开始迭代过滤，初始样本量: {len(df_cleaned)}")

iteration = 0
while True:
    iteration += 1
    initial_count = len(df_cleaned)
    
    # 1. 过滤品牌
    brand_counts = df_cleaned['brand_name_en_new'].value_counts()
    keep_brands = brand_counts[brand_counts >= min_count].index
    df_cleaned = df_cleaned[df_cleaned['brand_name_en_new'].isin(keep_brands)]
    
    # 2. 过滤品类
    cat_counts = df_cleaned['category_name_cn'].value_counts()
    keep_cats = cat_counts[cat_counts >= min_count].index
    df_cleaned = df_cleaned[df_cleaned['category_name_cn'].isin(keep_cats)]
    
    # 检查样本量是否还在变化
    final_count = len(df_cleaned)
    print(f"第 {iteration} 轮过滤完成，剩余样本: {final_count}")
    
    if final_count == initial_count:
        print("数据已达到稳定状态，停止迭代。")
        break

开始迭代过滤，初始样本量: 100787
第 1 轮过滤完成，剩余样本: 97255
第 2 轮过滤完成，剩余样本: 97126
第 3 轮过滤完成，剩余样本: 97126
数据已达到稳定状态，停止迭代。


In [67]:
# 提取时间特征
df_cleaned['month'] = df_cleaned['date'].dt.month
df_cleaned['day_of_month'] = df_cleaned['date'].dt.day

In [68]:
# 匹配活动日期
df_cleaned['is_activity_day'] = df_cleaned['date'].isin(activity_timeline['date']).astype(int)

In [69]:
# 对类别与品牌做encoding
le_cat = LabelEncoder()
le_brand = LabelEncoder()
df_cleaned['category_encoded'] = le_cat.fit_transform(df_cleaned['category_name_cn'])
df_cleaned['brand_encoded'] = le_brand.fit_transform(df_cleaned['brand_name_en_new'])

In [70]:
df_cleaned

,date,base_sku_id,category_name_cn,brand_name_en_new,sku_sale_amt,sku_sale_num,biz_total_discount_amt,month,day_of_month,is_activity_day,category_encoded,brand_encoded
0,2025-09-01,SKU1003,苏打汽水,Brand143,20.190,6,0.300,9,1,1,207,42
1,2025-09-01,SKU1007,混合香料粉,Brand164,37.770,2,6.150,9,1,1,146,62
2,2025-09-01,SKU1014,其它口味汽水,Brand55,8.595,4,1.800,9,1,1,29,331
3,2025-09-01,SKU1015,巧克力糖,Brand73,20.055,3,3.600,9,1,1,82,350
4,2025-09-01,SKU1017,甜饼干,Brand295,17.415,1,1.875,9,1,1,174,182
...,...,...,...,...,...,...,...,...,...,...,...,...
100782,2025-11-30,SKU993,燕麦,Brand292,13.875,1,6.300,11,30,1,156,180
100783,2025-11-30,SKU994,混合口味汽水,Brand293,12.375,3,0.000,11,30,1,143,181
100784,2025-11-30,SKU995,牛奶巧克力,Brand26,115.665,25,20.475,11,30,1,160,150
100785,2025-11-30,SKU996,袋装薯片/条,Brand13,20.625,3,4.950,11,30,1,220,31


In [71]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 97126 entries, 0 to 100786
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    97126 non-null  datetime64[ns]
 1   base_sku_id             97126 non-null  object        
 2   category_name_cn        97126 non-null  object        
 3   brand_name_en_new       97126 non-null  object        
 4   sku_sale_amt            97126 non-null  float64       
 5   sku_sale_num            97126 non-null  int64         
 6   biz_total_discount_amt  97126 non-null  float64       
 7   month                   97126 non-null  int64         
 8   day_of_month            97126 non-null  int64         
 9   is_activity_day         97126 non-null  int32         
 10  category_encoded        97126 non-null  int32         
 11  brand_encoded           97126 non-null  int32         
dtypes: datetime64[ns](1), float64(2), int32(3), i

In [72]:
df_cleaned['category_encoded'].value_counts()

220    5366
72     3819
188    2930
60     2681
167    2600
       ... 
171      38
96       37
103      37
54       35
25       33
Name: category_encoded, Length: 245, dtype: int64

In [73]:
df_cleaned['brand_encoded'].value_counts()

186    2643
348    2125
340    1931
78     1880
305    1830
       ... 
185      31
332      31
330      31
303      31
315      30
Name: brand_encoded, Length: 373, dtype: int64

In [19]:
# 目标变量 Y
Y = np.log1p(df_cleaned['sku_sale_amt']).values 
# 处理变量 T
T = np.log1p(df_cleaned['biz_total_discount_amt']).values

In [20]:
# 特征 X (用于异质性分析，即活动如何改变优惠效果)
X = df_cleaned[['is_activity_day']].values

In [21]:
# 控制变量 W (用于消除偏差的全局背景)
W_cols = [ 'category_encoded', 'brand_encoded', 'month', 'day_of_month']
W = df_cleaned[W_cols].values

In [22]:
from sklearn.preprocessing import StandardScaler

# 对数值型特征进行缩放
scaler = StandardScaler()
W_scaled = W.copy()
W_scaled = scaler.fit_transform(W)

In [23]:
# 合并 X 和 W 作为全量协变量供第一阶段模型使用
# R-Learner 会自动处理 X 中的异质性
X_all = np.hstack([X, W_scaled])

In [24]:
# --- Step 2: 模型配置 ---
# 使用 LightGBM 作为基础学习器
# 能够处理非线性关系，比如流量对销量的指数级拉动
r_learner = BaseRRegressor(
    learner=LGBMRegressor(n_estimators=30, max_depth=3, learning_rate=0.1, n_jobs=-1),
    random_state=42
)

In [25]:
# --- Step 3: 时序交叉验证训练 ---
tscv = TimeSeriesSplit(n_splits=3)
effects = []

for train_idx, test_idx in tscv.split(X_all):
    # 训练模型：注意 R-Learner 会执行“双重残差化”
    r_learner.fit(X=X_all[train_idx], treatment=T[train_idx], y=Y[train_idx])
    
    # 预测测试集的处理效应 (CATE)
    # 这代表：在当前环境下，额外投入 1 元优惠预期能带来的销量增长率
    test_effect = r_learner.predict(X_all[test_idx])
    effects.append(test_effect)

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(
C:\U

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppDa

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppDa

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppDa

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppDa

...

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppDa

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppDa

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\U

C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppData\Roaming\Python\Python39\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\eason\AppDa

KeyboardInterrupt: 

In [ ]:
df_test = sku_daily_sales.iloc[test_idx].copy()
df_test['estimated_discount_effect'] = test_effect

In [ ]:
# 计算平均处理效应 (ATE)
summary = df_test.groupby('is_activity_day')['estimated_discount_effect'].mean()
print("不同活动状态下的优惠边际收益：")
print(summary)